# Integración de un modelo ML en Flask para detectar URLs spam

## Objetivo

En este proyecto he reutilizado un modelo de `Machine Learning` entrenado previamente para clasificar URLs como `spam` o `no spam`.

El modelo proviene de mi proyecto anterior de NLP:

https://github.com/Cerco01/nlp-4geeks-project

En este notebook no voy a repetir todo el entrenamiento. El objetivo es comprobar que el modelo guardado funciona  y dejar claro cómo debe integrarse en una aplicación `Flask` desplegable en `Render`.


## 0. Configuración inicial


In [6]:
import joblib
import pandas as pd
import re

SEPARATOR = " " + 120 * '-' + " "

## 1. Carga del modelo final


In [8]:
model_path = "../models/svm_url_spam_classifier.joblib"
model = joblib.load(model_path)

print(f"Modelo cargado desde: {model_path}")
print(model)

Modelo cargado desde: ../models/svm_url_spam_classifier.joblib
Pipeline(steps=[('tfidf', TfidfVectorizer(min_df=2)),
                ('svm', SVC(C=10, kernel='linear', random_state=42))])


## 2. Contrato de inferencia

URL escrita por el usuario → `preprocess_url()` → modelo cargado → resultado `spam` / `no spam`


In [9]:
def preprocess_url(url):
    url = url.lower()
    tokens = re.split(r"[^a-zA-Z0-9]+", url)
    tokens = [token for token in tokens if token]
    return " ".join(tokens)


def predict_url(url):
    processed_url = preprocess_url(url)
    prediction = model.predict([processed_url])[0]
    return "spam" if prediction else "no spam"

## 3. Prueba rápida de predicción


In [10]:
sample_urls = [
    "https://www.reuters.com/investigates/special-report/health-coronavirus-britain-pub/",
    "https://briefingday.us8.list-manage.com/unsubscribe",
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ",
    "https://free-money-prize.example.com/winner/login",
]

results = pd.DataFrame({
    "url": sample_urls,
    "processed_url": [preprocess_url(url) for url in sample_urls],
    "prediction": [predict_url(url) for url in sample_urls],
})

display(results)

,url,processed_url,prediction
0,https://www.reuters.com/investigates/special-r...,https www reuters com investigates special rep...,no spam
1,https://briefingday.us8.list-manage.com/unsubs...,https briefingday us8 list manage com unsubscribe,spam
2,https://www.youtube.com/watch?v=dQw4w9WgXcQ,https www youtube com watch v dqw4w9wgxcq,no spam
3,https://free-money-prize.example.com/winner/login,https free money prize example com winner login,no spam
